# Re-verify Condenser Fouling, Liquid-Line Restriction, and Isolation Forest
# After the build_feature_table() Capacity Bug Fix

## Purpose

Notebook 24 found and fixed a real order-of-operations bug in
build_feature_table() (stage-2 filtering was happening before segmented EWMA
smoothing, not after, contradicting notebook 01's original validated approach).
This affects every model that includes capacity as a feature:
condenser_fouling, liquidline_restriction, and the Isolation Forest.

This notebook re-runs their TimeSeriesSplit evaluation with the CORRECTED
feature, to confirm whether their previously-documented "stable"/"usable"
status still holds, rather than assuming it does just because the bug is fixed.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit  # noqa: E402
from src.features.build_features import build_feature_table  # noqa: E402

table = build_feature_table(
    baseline_path=str(ml_root / "data/raw/RTU_sim_baseline.csv"),
    fault_paths={
        "condfouling10": str(ml_root / "data/raw/RTU_sim_condfouling10.csv"),
        "condfouling20": str(ml_root / "data/raw/RTU_sim_condfouling20.csv"),
        "condfouling30": str(ml_root / "data/raw/RTU_sim_condfouling30.csv"),
        "condfouling40": str(ml_root / "data/raw/RTU_sim_condfouling40.csv"),
        "condfouling50": str(ml_root / "data/raw/RTU_sim_condfouling50.csv"),
    },
    pressure_temp_cols=("RTU_REFG_COND_PRES", "RTU_REFG_COND_TEMP"),
)

feature_cols = ["RTU_REFG_COND_PRES_residual", "RTU_REFG_COND_TEMP_residual", "RTU_TOT_CAPA_ewma30_segmented_residual"]
X_all = table[feature_cols].values
y_all = table["label"].values

tscv = TimeSeriesSplit(n_splits=5)
print("=== Condenser fouling, TimeSeriesSplit, WITH CORRECTED capacity feature ===")
for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    report = classification_report(y_te, y_pred, target_names=["baseline", "condfouling"], output_dict=True)
    print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
          f"baseline precision={report['baseline']['precision']:.2f}, "
          f"condfouling recall={report['condfouling']['recall']:.2f}")

=== Condenser fouling, TimeSeriesSplit, WITH CORRECTED capacity feature ===
Fold 1: baseline recall=1.00, baseline precision=0.95, condfouling recall=0.99
Fold 2: baseline recall=1.00, baseline precision=0.93, condfouling recall=0.98
Fold 3: baseline recall=1.00, baseline precision=0.89, condfouling recall=0.97
Fold 4: baseline recall=1.00, baseline precision=0.85, condfouling recall=0.97
Fold 5: baseline recall=1.00, baseline precision=0.76, condfouling recall=0.94



## Condenser fouling: conclusion holds (stable), but precision drift is now larger

| Fold | Original (buggy capacity) | Corrected capacity |
|---|---|---|
| Baseline recall | 0.99-1.00 | 1.00 (all folds) |
| Baseline precision | 0.96 → 0.88 | 0.95 → 0.76 |

**Recall is actually slightly better with the fix** (perfect 1.00 across all 5
folds, vs. 0.99-1.00 before) - reassuring, the core "stable, no collapse"
finding holds and even strengthens slightly.

**Precision drifts further with the corrected feature** (down to 0.76 by fold 5,
vs. 0.88 before) - a real, honest change. This makes sense: the buggy version's
capacity feature was accidentally smoothed across separate real stage-2
sessions, which may have (coincidentally) suppressed some of capacity's real
signal noise; the corrected version, more faithfully representing real
session-to-session variation, shows a bit more of that natural variability,
translating to a few more false alarms later in the simulated year.

**Status unchanged**: still "Usable" per the registry, but the precision
caveat should be updated in FINAL_MODEL_METRICS.md to reflect the corrected,
somewhat wider drift (0.76 vs 0.88 at the low end).


## Liquid-line restriction: same check, with corrected capacity feature

In [2]:
table_ll = build_feature_table(
    baseline_path=str(ml_root / "data/raw/RTU_sim_baseline.csv"),
    fault_paths={
        "liquidpipe01bar": str(ml_root / "data/raw/RTU_sim_liquidpipe01bar.csv"),
        "liquidpipe04bar": str(ml_root / "data/raw/RTU_sim_liquidpipe04bar.csv"),
        "liquidpipe08bar": str(ml_root / "data/raw/RTU_sim_liquidpipe08bar.csv"),
        "liquidpipe10bar": str(ml_root / "data/raw/RTU_sim_liquidpipe10bar.csv"),
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_REFG_DISC_PRES"),
)

feature_cols_ll = [
    "RTU_REFG_SUCT_PRES_residual", "RTU_REFG_SUCT_TEMP_residual",
    "RTU_REFG_DISC_PRES_residual", "RTU_TOT_CAPA_ewma30_segmented_residual",
]
X_all_ll = table_ll[feature_cols_ll].values
y_all_ll = table_ll["label"].values

print("=== Liquid-line restriction, TimeSeriesSplit, WITH CORRECTED capacity feature ===")
for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all_ll), start=1):
    X_tr, X_te = X_all_ll[train_idx], X_all_ll[test_idx]
    y_tr, y_te = y_all_ll[train_idx], y_all_ll[test_idx]
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    report = classification_report(y_te, y_pred, target_names=["baseline", "liquidline"], output_dict=True)
    print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
          f"baseline precision={report['baseline']['precision']:.2f}, "
          f"liquidline recall={report['liquidline']['recall']:.2f}")

=== Liquid-line restriction, TimeSeriesSplit, WITH CORRECTED capacity feature ===
Fold 1: baseline recall=0.99, baseline precision=0.98, liquidline recall=1.00
Fold 2: baseline recall=1.00, baseline precision=0.97, liquidline recall=0.99
Fold 3: baseline recall=0.99, baseline precision=0.95, liquidline recall=0.99
Fold 4: baseline recall=1.00, baseline precision=0.93, liquidline recall=0.98
Fold 5: baseline recall=1.00, baseline precision=0.91, liquidline recall=0.97


## Liquid-line restriction: conclusion holds, small precision-drift widening

| Fold | Original (buggy capacity) | Corrected capacity |
|---|---|---|
| Baseline recall | 0.99-1.00 | 0.99-1.00 (unchanged) |
| Baseline precision | 0.98 → 0.96 | 0.98 → 0.91 |

Recall essentially unchanged. Precision drifts a bit more with the corrected
feature (down to 0.91 vs. 0.96 originally) - same pattern and same explanation
as condenser fouling. Status unchanged: still "Usable," stable, no
collapse - but the documented precision range should be updated to reflect
the corrected, slightly wider drift.

## Isolation Forest: same check, with corrected capacity feature

Rebuilding the full 6-fault feature table (same construction as notebook 18)
with the corrected capacity feature, re-checking false-positive rate and
detection-rate gradient.

In [3]:
from sklearn.ensemble import IsolationForest

all_fault_paths_check = {
    "undercharge10": str(ml_root / "data/raw/RTU_sim_undercharge10.csv"),
    "undercharge15": str(ml_root / "data/raw/RTU_sim_undercharge15.csv"),
    "undercharge20": str(ml_root / "data/raw/RTU_sim_undercharge20.csv"),
    "overcharge10": str(ml_root / "data/raw/RTU_sim_overcharge10.csv"),
    "overcharge15": str(ml_root / "data/raw/RTU_sim_overcharge15.csv"),
    "overcharge20": str(ml_root / "data/raw/RTU_sim_overcharge20.csv"),
    "condfouling10": str(ml_root / "data/raw/RTU_sim_condfouling10.csv"),
    "condfouling20": str(ml_root / "data/raw/RTU_sim_condfouling20.csv"),
    "condfouling30": str(ml_root / "data/raw/RTU_sim_condfouling30.csv"),
    "condfouling40": str(ml_root / "data/raw/RTU_sim_condfouling40.csv"),
    "condfouling50": str(ml_root / "data/raw/RTU_sim_condfouling50.csv"),
    "evapfouling10": str(ml_root / "data/raw/RTU_sim_evapfouling10.csv"),
    "evapfouling20": str(ml_root / "data/raw/RTU_sim_evapfouling20.csv"),
    "evapfouling30": str(ml_root / "data/raw/RTU_sim_evapfouling30.csv"),
    "evapfouling40": str(ml_root / "data/raw/RTU_sim_evapfouling40.csv"),
    "evapfouling50": str(ml_root / "data/raw/RTU_sim_evapfouling50.csv"),
    "liquidpipe01bar": str(ml_root / "data/raw/RTU_sim_liquidpipe01bar.csv"),
    "liquidpipe04bar": str(ml_root / "data/raw/RTU_sim_liquidpipe04bar.csv"),
    "liquidpipe08bar": str(ml_root / "data/raw/RTU_sim_liquidpipe08bar.csv"),
    "liquidpipe10bar": str(ml_root / "data/raw/RTU_sim_liquidpipe10bar.csv"),
    "suctionpipe01bar": str(ml_root / "data/raw/RTU_sim_suctionpipe01bar.csv"),
    "suctionpipe03bar": str(ml_root / "data/raw/RTU_sim_suctionpipe03bar.csv"),
    "suctionpipe06bar": str(ml_root / "data/raw/RTU_sim_suctionpipe06bar.csv"),
    "suctionpipe09bar": str(ml_root / "data/raw/RTU_sim_suctionpipe09bar.csv"),
}

table_if = build_feature_table(
    baseline_path=str(ml_root / "data/raw/RTU_sim_baseline.csv"),
    fault_paths=all_fault_paths_check,
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP"),
)

feature_cols_if = ["RTU_REFG_SUCT_PRES_residual", "RTU_REFG_SUCT_TEMP_residual", "RTU_TOT_CAPA_ewma30_segmented_residual"]
baseline_only_if = table_if[table_if["label"] == 0].sort_values("Datetime")
cutoff_if = baseline_only_if["Datetime"].min() + (baseline_only_if["Datetime"].max() - baseline_only_if["Datetime"].min()) * 0.8
train_baseline_if = baseline_only_if[baseline_only_if["Datetime"] < cutoff_if]
test_baseline_if = baseline_only_if[baseline_only_if["Datetime"] >= cutoff_if]

iso_check = IsolationForest(contamination=0.01, random_state=42, n_estimators=100)
iso_check.fit(train_baseline_if[feature_cols_if])

fpr_check = (iso_check.predict(test_baseline_if[feature_cols_if]) == -1).mean()
print(f"False positive rate, WITH CORRECTED capacity feature: {fpr_check:.1%}")

strong_fault_check = table_if[table_if["source_file"] == "suctionpipe09bar"]
weak_fault_check = table_if[table_if["source_file"] == "overcharge10"]
print(f"Strong-fault detection: {(iso_check.predict(strong_fault_check[feature_cols_if]) == -1).mean():.1%}")
print(f"Weak-fault detection: {(iso_check.predict(weak_fault_check[feature_cols_if]) == -1).mean():.1%}")

False positive rate, WITH CORRECTED capacity feature: 2.9%
Strong-fault detection: 100.0%
Weak-fault detection: 1.2%


## Isolation Forest: genuine improvement with corrected capacity feature

| Metric | Original (buggy capacity) | Corrected capacity |
|---|---|---|
| False positive rate | 6.1% | 2.9% |
| Strong-fault detection | 99-100% | 100.0% |
| Weak-fault detection | <0.25 (varies) | 1.2% |

**A genuine, real improvement** - false-positive rate roughly halved, with no
loss of strong-fault detection. Makes sense given the direction: the buggy
version's capacity feature was less faithful to real stage-2-session-to-session
variation, and the Isolation Forest (unlike the two supervised classifiers,
which showed slightly WORSE precision with the fix) benefits from a cleaner
baseline representation for its unsupervised anomaly threshold. Weak-fault
detection also dropped, consistent with the same broad pattern as the other
two rechecked models - the corrected feature doesn't change which faults are
inherently easy/hard to detect, just refines the baseline/detection boundary.

**Status unchanged, actually strengthened**: "Usable with caveat," now with a
lower false-positive rate than previously documented - should be updated in
FINAL_MODEL_METRICS.md as a genuine improvement, not just a correction.

## Summary: re-verification after the capacity-feature bug fix

All 3 affected models (condenser fouling, liquid-line restriction, Isolation
Forest) re-evaluated with the corrected feature. None flipped to unusable:

- **Condenser fouling**: still stable (recall 1.00 all folds, even slightly
  better than before). Precision drift widened (0.95→0.76 vs originally
  0.96→0.88) - a real, honest change to document.
- **Liquid-line restriction**: still stable (recall unchanged). Precision
  drift modestly widened (0.98→0.91 vs originally 0.98→0.96).
- **Isolation Forest**: genuine improvement - false-positive rate roughly
  halved (6.1%→2.9%), no loss of strong-fault detection.

**Overall conclusion**: the bug fix did not invalidate any prior deployment
decision, but DID change the precise numbers - FINAL_MODEL_METRICS.md and
MODEL_RESULTS_LOG.md must be updated to reflect the corrected values, not
left showing numbers computed on the buggy feature.